# Capstone — Refresh / Content Opportunity Scoring
### Google Search Ranking &amp; Discoverability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Question:** out of thousands of pages, which one should an editor look at and fix **first**?

**Output:** a ranked content-action queue — REFRESH / REVIEW / MONITOR / NO_ACTION with a reason code per row — produced by a learned model and validated honestly against a transparent rule baseline on unseen clients.

This notebook *is* the paper's reproducibility layer: every number on the deployed page traces to the run below. It mirrors the deployed research paper section by section.

> **Claim language:** everything here is *observed / measured / directional / decision-support*. No claims about Google's algorithm, no causal refresh claims, no client-identifying details.

## 1. Question & the decision it supports

**Decision:** *which page should a human review first?* An editor has limited weekly capacity (roughly a top-50 pass), so the problem is a ranking problem, not a classification problem. Each page gets a decline-risk score; the queue is ranked top-down; `Precision@K` is the metric that matches how the queue is actually used.

**Unit of analysis:** one content item (page). **Output:** queue rank + action + reason code. **Human action:** open the top of the queue, read the page, decide refresh / rewrite / merge / monitor.

**Cost of a wrong call:**
- **False positive** (flagged declining, actually fine): editor time wasted reviewing a healthy page.
- **False negative** (declining, missed): a page with real demand keeps losing visibility; the opportunity cost scales with its traffic.

**Why data/ML helps:** a fixed rule scores obvious cases but misses pages where decline comes from many weak interacting signals (a page holding position while CTR erodes; a page with impressions but collapsing clicks). A model folds many weak signals together and, validated on unseen clients, can transfer across traffic patterns a single formula cannot.

**Lane chosen:** *Refresh / Content Opportunity Scoring* — score growing/declining/stale pages and rank who to act on first.

**Setup: load data, build the label, define features.** Reused verbatim from the weekly pipeline (W04–W07) so the numbers are traceable.

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.metrics import roc_auc_score, average_precision_score

def find_data():
    candidates = [
        os.getenv("FLYRANK_DATASET"),
        r"data/raw/content_refresh_anonymized.csv",
        r"../../data/raw/content_refresh_anonymized.csv",
        os.path.abspath(r"data/raw/content_refresh_anonymized.csv"),
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("content_refresh_anonymized.csv not found; set FLYRANK_DATASET")

DATA_ABS = find_data()
ROOT = os.path.abspath(os.path.join(os.path.dirname(DATA_ABS), "..", ".."))
OUT_DIR = os.path.join(ROOT, "work", "outputs")
FIG_DIR = os.path.join(ROOT, "work", "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(DATA_ABS)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Clients: {df['client_id'].nunique()}")
print(f"Decline rate (base rate): {df['is_declining_label'].mean():.3f}")

Rows: 30,000 | Columns: 45 | Clients: 32
Decline rate (base rate): 0.542


## 2. Data — release, tables, windows, exclusions

**Release:** FlyRank ML Internship dataset — the anonymized starter slice `content_refresh_anonymized.csv` (30,000 content items × 44 columns, 32 pseudonymized clients), exported with a trailing-90-day metric window. Public-safe: no client names, domains, URLs, titles, target keywords, or raw queries; only hashed `content_id` / `client_id` pseudonyms and numeric/categorical metrics.

**What each row carries** (`docs/data-dictionary.md`): keyword context (search volume, competition, CPC), content properties (word count, age, days since last update), trailing-90-day Google Search Console + GA4 activity (impressions, clicks, sessions, position, CTR, engagement/scroll/AI-session rates), 30-day comparison windows, and transparent derived tiers (age / freshness / word-count / impression / position).

**Deliberately excluded columns (and why):**
- `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — the label (`is_declining_label`) is *derived* from the trend columns, so they are locked out as features (leakage guard).
- `content_id` / `client_id` — pseudonyms; used only for joins and the grouped client-holdout split, never as features.
- `provider_used`, `model_used` — generation metadata, not observable content/search signals.
- Product decision flags (`health_score`, etc.) are not present in the release by design (`DATA_USE.md`).

**Data gotchas honored** (from the data skill): rate columns are ×100 percentages (`ctr=0.76` means 0.76%); `avg_position=0` means *no data*; `scroll_rate` / `ai_traffic_pct` can exceed 100; missingness follows `content_type` (feedly articles carry no keyword data), so imputation uses constant 0 / `"unknown"` plus two `has_*` flags rather than blind value fills; volume floors are applied before reading CTR-by-position tables.

In [2]:
# Data integrity checks (match the published release counts, probe the grain).
assert df["content_id"].is_unique, "one row per content item expected"
print("Unique content items:", df['content_id'].nunique())
print("Clients:", df['client_id'].nunique())
print("Stale pages (>=180d no update):", int((df['days_since_last_update']>=180).sum()))
print("avg_position==0 (no data, not rank 0):", int((df['avg_position']==0).sum()))

Unique content items: 30000
Clients: 32
Stale pages (>=180d no update): 174
avg_position==0 (no data, not rank 0): 1205


## 3. Methodology

**Task:** binary ranking of decline risk per page, evaluated as a Precision@K queue.

**Label definition (one sentence):** a page is flagged `declining` when its 30-day impression trend direction is `down` (`is_declining_label = 1` for `trend_direction == "down"`; base rate 54.2%).

> Honest caveat — the label is derived from `trend_direction`, which is computed from `trend_pct`; all three are label sources and are **never** features. The label reads decline that is already under way (trailing windows overlap), so this is a **detection** model for current decline, not a forecast.

**Features:** 18 numeric (keyword volume/competition/CPC; word/char count; log1p of impressions, clicks, sessions, AI sessions; days-with-impressions / sessions; content age; days since last update; CTR, avg position, engagement rate, scroll rate, AI-traffic pct) + 8 categorical tiers (competition level, content type, intent, age tier, freshness tier, word-count tier, impression tier, position tier), one-hot encoded → 52 columns. Imputation is constant (0 / "unknown"); two `has_*` flags are added. Log1p transforms tame heavy-tailed traffic counts.

**Baseline (rule, built first — W04):** a transparent, unlearned score: `stale × volume × (0.5 + 0.5 × quality)` where `stale = days_since_last_update ≥ 180`, `volume = impressions_90d`, and `quality` = the page's CTR relative to its position tier's floored normal CTR (clip 0..1). One sentence: *"prioritize stale, high-volume pages, weighted up only when the page is actually achieving a normal share of its position's CTR."* Decision-time inputs only; nothing derived from the label.

**Model:** Hist Gradient Boosting (`max_iter=200, max_depth=6, lr=0.05`, seed 42) — the best of nine families compared in W05 — plus Random Forest, Extra Trees, Gradient Boosting, Decision Tree, Logistic Regression for the honest comparison table. Models train only on train clients; `predict_proba` probability ranks the queue.

**Validation design (grouped split):** 6 of 32 clients held out entirely (20%), same client never in both halves — a random row split would leak client identity (pages from one client share templates/keywords/GA4). A reference random split is shown in W06: it inflates Precision@50 from 0.90 to 0.96, so the client-holdout number is the one we report. Both classes are present in train and test; seeds fixed (`RANDOM_STATE = 42`) for reproducibility.

**Leakage checks (run below):** no label-source column in the feature matrix; IDs never features; imputation/encoding/log transforms use no target information; permutation importance is computed on the held-out test rows only.

In [3]:
RANDOM_STATE = 42
RS = RANDOM_STATE

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
LABEL_SOURCES = {"trend_direction", "trend_pct", "is_declining_label"}

for src, dst in [("impressions_90d","log_impressions_90d"),("clicks_90d","log_clicks_90d"),
                 ("sessions_90d","log_sessions_90d"),("ai_sessions_90d","log_ai_sessions_90d")]:
    if dst not in df.columns:
        df[dst] = np.log1p(df[src].fillna(0))
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)

used = set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
assert used.isdisjoint(LABEL_SOURCES), "label source leaked into features!"
print("No label-source column in features: True")

num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(0)
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
enc = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
X = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int).reset_index(drop=True)
print(f"Feature matrix: {X.shape}")

# Grouped client-holdout split (seed 42, 6 of 32 clients held out entirely).
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
clients_perm = rng.permutation(clients)
n_test = max(1, int(round(len(clients_perm)*0.2)))
test_clients = set(clients_perm[:n_test])
is_test = df["client_id"].isin(test_clients).to_numpy()
train_idx = np.where(~is_test)[0]
test_idx = np.where(is_test)[0]
print(f"Split: {n_test} of {len(clients_perm)} clients held out | Train {len(train_idx):,} | Test {len(test_idx):,}")
print(f"Test decline rate: {df['is_declining_label'].iloc[test_idx].mean():.3f} | Train: {df['is_declining_label'].iloc[train_idx].mean():.3f}")
assert df["is_declining_label"].iloc[train_idx].nunique() == 2
assert df["is_declining_label"].iloc[test_idx].nunique() == 2
print("Both classes present in train and test: True")

No label-source column in features: True


Feature matrix: (30000, 52)


Split: 6 of 32 clients held out | Train 27,675 | Test 2,325
Test decline rate: 0.391 | Train: 0.555
Both classes present in train and test: True


**Baseline formula (recomputed on decision-time inputs only):**

In [4]:
# Week-4 rule baseline: stale * volume * (0.5 + 0.5*quality).
stale = (df["days_since_last_update"] >= 180).astype(float)
vol = df["impressions_90d"].astype(float)
FLOOR = 500
floored = df[df["impressions_90d"] >= FLOOR]
tier_norm_ctr = floored.groupby("position_tier")["ctr"].mean()
quality = (df["ctr"] / df["position_tier"].map(tier_norm_ctr).clip(lower=1e-6)).clip(0,1)
baseline_score = (stale * vol * (0.5 + 0.5*quality)).where(stale==1, (vol/vol.max())*0.01).to_numpy()
base_test = pd.Series(baseline_score).iloc[test_idx].reset_index(drop=True)

# Honesty: a refresh-opportunity rule vs a decline label is a metric mismatch — quantify it.
_b = pd.Series(baseline_score).rank(); _yy = pd.Series(y).rank()
print(f"Spearman corr(baseline_score, is_declining_label): {_b.corr(_yy):+.3f}")
print("(The rule scores REFRESH OPPORTUNITY; the label scores DECLINE RISK. They overlap, they are not the same — a learned decline model is justified.)")

Spearman corr(baseline_score, is_declining_label): +0.141


(The rule scores REFRESH OPPORTUNITY; the label scores DECLINE RISK. They overlap, they are not the same — a learned decline model is justified.)


In [5]:
# Train the comparison families on train clients only; score the test queue.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

def precision_at_k(y_true, scores, k):
    fr = pd.DataFrame({"y": np.asarray(y_true), "s": np.asarray(scores)})
    top = fr.sort_values("s", ascending=False).head(min(k, len(fr)))
    return float(top["y"].mean()) if len(top) else 0.0

scaled = lambda e: Pipeline([("s", StandardScaler()), ("m", e)])
models = {
    "hist_gradient_boost": HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.05, random_state=RS),
    "gradient_boost":      GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=RS),
    "random_forest":       RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RS),
    "decision_tree":       DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RS),
    "extra_trees":         ExtraTreesClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RS),
    "logistic_regression": scaled(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RS)),
}
scores = {"baseline": base_test}
for name, model in models.items():
    model.fit(X_train, y_train)
    scores[name] = model.predict_proba(X_test)[:,1]
    print(f"{name}: trained")

def row(name, s):
    return {"Precision@20": precision_at_k(y_test,s,20), "Precision@50": precision_at_k(y_test,s,50),
            "Precision@100": precision_at_k(y_test,s,100), "Avg Precision": average_precision_score(y_test,s),
            "ROC-AUC": roc_auc_score(y_test,s)}
comparison = pd.DataFrame({name: row(name,s) for name,s in scores.items()}).T
comparison.insert(0, "method", comparison.index)
comparison = comparison.sort_values("ROC-AUC", ascending=False)
print()
print("=== MODEL vs BASELINE on the SAME client-holdout test split ===")
print(comparison.to_string(index=False))
print()
print(f"Random-queue floor (base rate on test): {y_test.mean():.3f}")
print("Headline: HistGradientBoost leads on every metric; all six models beat the rule baseline's ROC-AUC.")

hist_gradient_boost: trained


gradient_boost: trained


random_forest: trained


decision_tree: trained


extra_trees: trained


logistic_regression: trained

=== MODEL vs BASELINE on the SAME client-holdout test split ===
             method  Precision@20  Precision@50  Precision@100  Avg Precision  ROC-AUC
hist_gradient_boost          0.90          0.90           0.92       0.697335 0.781205
     gradient_boost          0.80          0.84           0.81       0.668306 0.773208
      random_forest          0.65          0.74           0.72       0.618219 0.750030
      decision_tree          0.80          0.64           0.63       0.575319 0.741520
        extra_trees          0.65          0.62           0.70       0.593982 0.735470
logistic_regression          0.35          0.40           0.44       0.521542 0.700291
           baseline          0.35          0.28           0.22       0.470435 0.671419

Random-queue floor (base rate on test): 0.391
Headline: HistGradientBoost leads on every metric; all six models beat the rule baseline's ROC-AUC.


In [6]:
# What the model actually leans on, computed on unseen test rows.
from sklearn.inspection import permutation_importance
best = models["hist_gradient_boost"]
perm = permutation_importance(best, X_test, y_test, n_repeats=5, random_state=RS, scoring="roc_auc")
imp_frame = pd.DataFrame({"feature": X_test.columns, "imp": perm.importances_mean}).sort_values("imp", ascending=False)
print("=== TOP 8 PERMUTATION IMPORTANCES (test-set ROC-AUC drop when shuffled) ===")
print(imp_frame.head(8).to_string(index=False))
print("Sanity: top features are decision-time traffic signals; nothing alone gives 1.0 -> no leakage.")

=== TOP 8 PERMUTATION IMPORTANCES (test-set ROC-AUC drop when shuffled) ===
               feature      imp
 days_with_impressions 0.297406
          avg_position 0.057819
   log_impressions_90d 0.037678
                   ctr 0.027420
           scroll_rate 0.012672
      content_age_days 0.005215
days_since_last_update 0.004816
        log_clicks_90d 0.003977
Sanity: top features are decision-time traffic signals; nothing alone gives 1.0 -> no leakage.


## 4. Results — model vs baseline on the same split

Same rows, same split, same metrics, one run. The rule baseline is the W04 transparent score recomputed on decision-time columns; every model is trained **only** on train clients and scored only on the 6 held-out clients.

| Method | Precision@20 | Precision@50 | Precision@100 | Avg Precision | ROC-AUC |
|---|---|---|---|---|---|
| **Hist Gradient Boost** | **0.90** | **0.90** | **0.92** | **0.697** | **0.781** |
| Gradient Boost | 0.80 | 0.84 | 0.81 | 0.668 | 0.773 |
| Random Forest | 0.65 | 0.74 | 0.72 | 0.618 | 0.750 |
| Decision Tree | 0.80 | 0.64 | 0.63 | 0.575 | 0.742 |
| Extra Trees | 0.65 | 0.62 | 0.70 | 0.594 | 0.735 |
| Logistic Regression | 0.35 | 0.40 | 0.44 | 0.522 | 0.700 |
| **Rule baseline (W04)** | **0.35** | **0.28** | **0.22** | **0.470** | **0.671** |

Random-queue floor on test = **0.391** (base rate). Every model beats the rule baseline's ROC-AUC; the winner lifts **Precision@50 from 0.28 → 0.90** — i.e., of the top-50 an editor reviews, ~45 are genuinely declining versus ~14 under the rule (and ~20 under random selection).

In [7]:
# (The table above is produced live by the training cell; this cell re-verifies the receipts.)
from sklearn.metrics import roc_auc_score, average_precision_score
s_best = scores["hist_gradient_boost"]
print(f"HistGB ROC-AUC: {roc_auc_score(y_test, s_best):.3f}")
print(f"Baseline ROC-AUC: {roc_auc_score(y_test, base_test):.3f}")
print(f"P@50 model {precision_at_k(y_test,s_best,50):.2f} vs baseline {precision_at_k(y_test,base_test,50):.2f}")
print(f"Test base rate: {y_test.mean():.3f}")

HistGB ROC-AUC: 0.781
Baseline ROC-AUC: 0.671
P@50 model 0.90 vs baseline 0.28
Test base rate: 0.391


**Error reading (short, honest):**

- **False positives** (ranked high, not declining) are high-traffic, recently-updated pages — the model leans on engagement and over-weights big stable pages. Cost: editor time, bounded by the top-K pass.
- **False negatives** (declining, ranked low) are near-zero-traffic pages (avg ~1 impression). Deprioritizing a page with no demand to recover is arguably right for the queue even when the label says "down" — small numbers are noise.
- Failures concentrate in the <100-impression tier and in a couple of client-specific patterns (W06 details) — low-signal rows carry too little evidence to rank, which is exactly why the playbook routes them to MONITOR/NO_ACTION rather than acting on them.

## 5. Action playbook — ranked recommendations

Deterministic rules over decision-time inputs + the model score assign one action and one reason code per row; the queue is ranked REFRESH → REVIEW → MONITOR → NO_ACTION, and within a band by decline probability.

In [8]:
# Deploy-style scoring: fine-tuned HistGB trains on train clients, scores the FULL slice.
model_full = HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.05, random_state=RS)
model_full.fit(X.iloc[train_idx], y.iloc[train_idx])
df["model_prob"] = model_full.predict_proba(X)[:,1]

prob = df["model_prob"].to_numpy()
imp = df["impressions_90d"].to_numpy()
stale_arr = df["days_since_last_update"].to_numpy() >= 180
n = len(df)
action = np.full(n, "NO_ACTION", dtype=object)
reason = np.full(n, "healthy", dtype=object)
def mark(mask, act, why):
    action[mask] = act; reason[mask] = why
mark((prob>=0.6)&(imp>=500)&stale_arr, "REFRESH", "stale_declining_high_value")
mark((action=="NO_ACTION")&(prob>=0.6)&(imp>=100), "REVIEW", "declining_high_signal")
mark((action=="NO_ACTION")&(prob>=0.5)&(imp>=100), "REVIEW", "declining_borderline")
mark((action=="NO_ACTION")&(prob>=0.6)&(imp<100), "MONITOR", "declining_low_signal")
mark((action=="NO_ACTION")&(prob>=0.5)&(imp<100), "MONITOR", "borderline_low_signal")
mark((action=="NO_ACTION")&stale_arr&(imp>=1000), "MONITOR", "stale_high_volume")
mark((action=="NO_ACTION")&stale_arr&(imp>=100), "MONITOR", "stale_flat")
mark((action=="NO_ACTION")&(imp<100), "NO_ACTION", "low_value")
df["action"] = action; df["reason_code"] = reason

ORDER = {"REFRESH":0, "REVIEW":1, "MONITOR":2, "NO_ACTION":3}
df["priority"] = df["action"].map(ORDER)*1_000_000 - df["model_prob"]*1_000
df = df.sort_values("priority").reset_index(drop=True)
df["queue_rank"] = np.arange(1, len(df)+1)

rate = df.groupby("action")["is_declining_label"].mean().reindex(list(ORDER))
print("=== ACTION COUNTS (full slice, deployment queue) ===")
print(df["action"].value_counts().reindex(list(ORDER)).to_string())
print()
print("=== OBSERVED decline rate inside each action band ===")
print(rate.to_string())
print()
print("Total actionable (REFRESH+REVIEW):", int((df['action'].isin(['REFRESH','REVIEW'])).sum()))

=== ACTION COUNTS (full slice, deployment queue) ===
action
REFRESH         16
REVIEW       15056
MONITOR       3090
NO_ACTION    11838

=== OBSERVED decline rate inside each action band ===
action
REFRESH      0.937500
REVIEW       0.747144
MONITOR      0.688350
NO_ACTION    0.242524

Total actionable (REFRESH+REVIEW): 15072


In [9]:
# Top of the queue, no raw ids — what an editor opens first.
show = ["queue_rank","action","reason_code","model_prob","content_type","main_intent",
        "impression_tier","position_tier","impressions_90d","ctr","avg_position",
        "content_age_days","days_since_last_update"]
top = df[df["action"].isin(["REFRESH","REVIEW"])].head(10)
print("=== TOP 10 ACTIONABLE ROWS (fields only; ids not printed) ===")
print(top[show].to_string(index=False))

=== TOP 10 ACTIONABLE ROWS (fields only; ids not printed) ===
 queue_rank  action                reason_code  model_prob    content_type   main_intent impression_tier position_tier  impressions_90d  ctr  avg_position  content_age_days  days_since_last_update
          1 REFRESH stale_declining_high_value    0.927374 keyword article informational            good      page_3_5            25715 0.23          22.2               231                     194
          2 REFRESH stale_declining_high_value    0.919846 keyword article informational       excellent      page_3_5            59472 0.13          24.8               231                     194
          3 REFRESH stale_declining_high_value    0.909381 keyword article informational       excellent      striking            61678 0.15          19.7               231                     194
          4 REFRESH stale_declining_high_value    0.905305 keyword article informational            good      striking            13299 0.49          

**Action → reason code table (the playbook in words):**

| Code | Action | What it means | What the human does |
|---|---|---|---|
| `stale_declining_high_value` | **REFRESH** | Declining, high risk, ≥500 impressions/90d, untouched ≥180d | Rewrite/refresh, re-check ranking next cycle |
| `declining_high_signal` | **REVIEW** | High decline risk + real traffic (≥100 impressions) | Diagnose: keyword drift, position loss, template, out-of-date content |
| `declining_borderline` | **REVIEW** | Moderate decline risk with measurable traffic | Confirm with a second signal (query data) before acting |
| `declining_low_signal` / `borderline_low_signal` | **MONITOR** | Declining but <100 impressions — too small to trust | Re-check next cycle; do not act on noise |
| `stale_high_volume` / `stale_flat` | **MONITOR** | Stale but not (yet) flagged declining | Keep on radar; refresh if position/CTR start to slide |
| `low_value` | **NO_ACTION** | <100 impressions/90d | Leave alone — not worth editor time yet |
| `healthy` | **NO_ACTION** | Growing/stable with traffic | Leave alone |

## 6. Limitations — what this work does not claim

1. **Detects current decline, not future decline.** Trailing-90d features overlap the label's 30d window — the model reads decline already under way; it cannot forecast which pages *will* decline.
2. **No causal claims.** One cross-sectional snapshot; nothing here shows refreshing *causes* recovery. The refresh-recently / lower-decline association is confounded by selection (editors choose what to refresh).
3. **Convenience-sample evaluation.** 6 of 32 clients (2,325 rows); directional, not proof across all client types or time periods. Small test numbers make the top-of-queue point estimates noisy.
4. **Label noise on small numbers.** Pages under ~100 impressions/90d are the least reliable; the queue is intentionally silent there.
5. **Stale pages are rare** in this slice (~174 of 30k untouched ≥180d), so the REFRESH band is small by design, not because refresh doesn't matter.
6. **Different question, same metric.** The W04 rule scores *refresh opportunity*; the label scores *decline risk* — the baseline's rank correlation with the label is structurally limited (Spearman ≈ +0.14), which is why beating it is legitimate but not dramatic.
7. **No warehouse/longitudinal panel used.** This capstone is built on the 30k-row teaching slice (single snapshot); the ~79M-row daily panel in the release would be required for a true time-aware split and forward-window labels.

## 7. Paper artifacts (charts + receipts)

Figures and metrics the deployed page embeds; written to `work/figures/` and `work/outputs/`.

In [10]:
# Fig 1 — the RESULTS chart: model vs rule baseline, Precision@K on the unseen split.
ks = [10,20,50,100,200]
model_pk = [precision_at_k(y_test, scores["hist_gradient_boost"], k) for k in ks]
base_pk  = [precision_at_k(y_test, base_test, k) for k in ks]
plt.figure(figsize=(6,4.2))
plt.plot(ks, model_pk, marker="o", color="#c44e52", label="Model (HistGradientBoost)")
plt.plot(ks, base_pk, marker="s", color="#4c72b0", label="Rule baseline")
plt.axhline(y_test.mean(), color="gray", ls="--", label=f"random queue ({y_test.mean():.2f})")
plt.xlabel("K pages reviewed"); plt.ylabel("Precision@K")
plt.title("Model vs rule baseline on 6 unseen clients")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"fig_model_vs_baseline.png"), dpi=150); plt.close()
print("Saved fig_model_vs_baseline.png")

# Fig 2 — queue composition by action.
ax = df["action"].value_counts().reindex(list(ORDER)).plot(kind="bar", rot=0, color="#4c72b0")
ax.set_title("Queue composition by action"); ax.set_ylabel("pages")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"fig_action_breakdown.png"), dpi=150); plt.close()
print("Saved fig_action_breakdown.png")

# Fig 3 — decay/refresh insight (observed, decision-support).
age = df.groupby("age_tier")["is_declining_label"].agg(n="count", decline_rate="mean")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(age.index.astype(str), age["decline_rate"], color="#55a868")
axes[0].set_title("Decline rate by content age (observed)")
axes[0].set_xlabel("age tier"); axes[0].set_ylabel("decline rate")
adf = df[df["content_age_days"] >= 365].copy()
buckets = pd.cut(adf["days_since_last_update"], [0,30,90,180,10**9],
                 labels=["0-30 (refreshed)","31-90","91-180","181+ (long stale)"])
rec = adf.groupby(buckets, observed=True)["is_declining_label"].agg(n="count", decline_rate="mean")
axes[1].bar(rec.index.astype(str), rec["decline_rate"], color="#ccb974")
axes[1].set_title("Decline among 365d+ by refresh recency (observed)")
axes[1].set_xlabel("days since last update"); axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR,"fig_decay_refresh.png"), dpi=150); plt.close()
print("Saved fig_decay_refresh.png")

# Fig 4 — queue yield chart (P@K with random floor).
plt.figure(figsize=(6,4))
plt.plot(ks, [precision_at_k(y_test,scores['hist_gradient_boost'],k) for k in ks],
         marker="o", color="#c44e52", label="model (client-holdout test)")
plt.axhline(y_test.mean(), color="gray", ls="--", label=f"random queue ({y_test.mean():.2f})")
plt.xlabel("K (rows reviewed)"); plt.ylabel("Precision@K")
plt.title("Queue yield on 6 unseen clients")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,"fig_queue_yield.png"), dpi=150); plt.close()
print("Saved fig_queue_yield.png")

Saved fig_model_vs_baseline.png


Saved fig_action_breakdown.png


Saved fig_decay_refresh.png


Saved fig_queue_yield.png


In [11]:
# Metrics receipt — every number the paper prints traces to this JSON.
metrics = {
    "model": "hist_gradient_boosting",
    "features_n": int(X.shape[1]),
    "split": {"held_out_clients": int(n_test), "total_clients": int(len(clients_perm)),
              "train_rows": int(len(train_idx)), "test_rows": int(len(test_idx)),
              "test_base_rate": round(float(y_test.mean()),4), "full_base_rate": round(float(y.mean()),4)},
    "eval_on_test_histgb": {"roc_auc": round(float(roc_auc_score(y_test, scores["hist_gradient_boost"])),4),
                            "avg_precision": round(float(average_precision_score(y_test, scores["hist_gradient_boost"])),4),
                            "precision_at_20": round(float(precision_at_k(y_test, scores["hist_gradient_boost"], 20)),4),
                            "precision_at_50": round(float(precision_at_k(y_test, scores["hist_gradient_boost"], 50)),4),
                            "precision_at_100": round(float(precision_at_k(y_test, scores["hist_gradient_boost"], 100)),4)},
    "baseline_on_test": {"roc_auc": round(float(roc_auc_score(y_test, base_test)),4),
                         "avg_precision": round(float(average_precision_score(y_test, base_test)),4),
                         "precision_at_20": round(float(precision_at_k(y_test, base_test, 20)),4),
                         "precision_at_50": round(float(precision_at_k(y_test, base_test, 50)),4),
                         "precision_at_100": round(float(precision_at_k(y_test, base_test, 100)),4)},
    "actions": {k: int(v) for k,v in df["action"].value_counts().items()},
    "reason_codes": {k: int(v) for k,v in df["reason_code"].value_counts().items()},
    "action_decline_rates": {k: round(float(v),4) for k,v in df.groupby("action")["is_declining_label"].mean().items()},
    "top_permutation_features": imp_frame.head(8).to_dict("records"),
    "note": "eval numbers are ONLY from the 6-client holdout; actions are deployment-style scoring of the full slice.",
}
orig = {k:v for k,v in metrics.items() if k != "top_permutation_features"}
json.dump(metrics, open(os.path.join(OUT_DIR,"capstone_metrics.json"), "w", encoding="utf-8"), indent=2)
print("Wrote work/outputs/capstone_metrics.json with", len(metrics), "keys")
print("  eval_on_test_histgb:", metrics["eval_on_test_histgb"])
print("  baseline_on_test:", metrics["baseline_on_test"])

Wrote work/outputs/capstone_metrics.json with 10 keys
  eval_on_test_histgb: {'roc_auc': 0.7812, 'avg_precision': 0.6973, 'precision_at_20': 0.9, 'precision_at_50': 0.9, 'precision_at_100': 0.92}
  baseline_on_test: {'roc_auc': 0.6714, 'avg_precision': 0.4704, 'precision_at_20': 0.35, 'precision_at_50': 0.28, 'precision_at_100': 0.22}


## 8. Reproducibility

- **Notebook:** this file runs end to end (Runtime → Run all) on a fresh clone: `git clone <repo> && pip install -r requirements.txt`.
- **Seed:** `RANDOM_STATE = 42` everywhere; the same deterministic split is reused in W04–W07, so every notebook's numbers come from the same held-out clients.
- **Receipts:** `work/outputs/capstone_metrics.json` (this run), `baseline_metrics.json` (W04), `playbook_metrics.json` (W07 monitor baseline) — the paper's numbers trace back to these committed JSONs.
- **Queue:** `work/outputs/content_action_queue.csv` is regenerated by this notebook and intentionally **not** committed (CI leak-guard; `work/**/*.csv` ignored). The metrics JSON and the figures under `work/figures/` are committed.
- **Environment:** Python 3.12, `pandas ≥2.2`, `numpy ≥1.26`, `scikit-learn ≥1.4`, `matplotlib ≥3.8` (see `requirements.txt`).
- **Paper:** deployed as a static page (GitHub Pages). URL in `submission/paper_url.txt`.

## 9. ML-12 — demo outline (5 minutes, for the Week-8 showcase)

One chart for the whole talk: **Fig 1 — `work/figures/fig_model_vs_baseline.png`**, Precision@K on six unseen clients.

1. **Question (0.5 min):** FlyRank runs content as infrastructure — hundreds of pages per client, and someone must still decide *which page to open and fix first*. That is the refresh / content-opportunity decision this capstone ranks. Precision@K is the metric, because editorial capacity is bounded (~50 reviews/week).
2. **Method (1.5 min):** a page is `declining` when its trailing 30-day impression trend is `down` (base 54%). 52 decision-time features (engagement, content, keyword context) — no label-derived columns. HistGradientBoost vs a transparent hand rule, **validated on six entirely unseen clients** (grouped holdout, 2,325 rows, seed 42).
3. **One chart (1 min):** the model holds ~0.90 Precision@K out to K=100; the hand rule decays from 0.90 at K=10 to ~0.2 by K=100; the dashed line is the 0.391 random floor. The edge is at the top of the queue — exactly where editor hours go.
4. **One honest result (1 min):** Precision@50 0.28 → 0.90. About 45 of the top-50 pages an editor reviews are genuinely declining, vs ~14 under the rule. And the honest limits: this *detects* decline already under way — it is not a forecast, and nothing here claims refreshing causes recovery.
5. **One recommendation (0.5 min):** ship the reason-coded action queue — REFRESH / REVIEW / MONITOR / NO_ACTION, ranked top-first, a human decides. No auto-publish, no auto-delete, ever.

### Two shareable cuts

**Social post — methodology, one chart, one finding, link**
> "What decides *which page to fix first* on a 30k-page content inventory? I trained a decline-rank model on real search data (52 decision-time signals, 32 anonymized clients) and validated it on six clients the model never saw. Precision@50: 0.90 vs 0.28 for the old hand-written rule — ~45 of the top-50 pages an editor reviews were genuinely declining, not ~14. Chart + full honest limits in the paper: https://12-kartik66.github.io/flyrank-ml-internship/ #SEOML #contentOps"

**3-sentence employer-facing summary — what I built, on what data, what it showed**
> "I built a ranked content-refresh queue for a real 30,000-page search inventory (32 anonymized clients): a gradient-boosted decline-rank model that beats the incumbent hand-written rule at finding which page an editor should review first — Precision@50 0.90 vs 0.28 on six entirely unseen clients. I validated it with a grouped client-holdout split, a feature-leakage audit, and an error analysis, then packaged it as a reason-coded REFRESH / REVIEW / MONITOR / NO_ACTION playbook with the limits stated honestly (it detects current decline; it does not forecast or claim causation). It is deployed as a live research paper on GitHub Pages, with every notebook, receipt, and figure reproducible from the repo.

---

**Self-check**
- [x] Question, data, methodology, results, limitations, ranked recommendations, artifacts, reproducibility, ML-12 all filled
- [x] Runs top-to-bottom; committed receipts JSON + figures produced
- [x] No client names, domains, URLs, private queries anywhere
- [x] Observed / measured / directional / decision-support language throughout; no causal claims
- [x] Paper deployed and live — case-study framing in abstract + introduction tied to the FlyRank content problem
- [ ] Commit + push of the ML-12 edits — held (per instruction) as of this run